### Estimador para Callaway e Sant'Anna (2021)

**ideia geral**

A ideia aqui é partir do estimador de Sant'Anna e Zhao (2020) e adaptá-lo para o caso de múltiplos períodos de tratamento, como proposto por Callaway e Sant'Anna (2021). Vamos utilizar o pacote em python do DoubleML para implementar o estimador. E compará-lo com o estimador `differences` do python.

**Objetivo do Projeto**

* Implementar o estimador de Callaway & Sant’Anna (2021) para múltiplos períodos de tratamento, adaptando o DoubleML (que já possui o estimador de Sant'Anna & Zhao, 2020).
* Comparar os resultados com o estimador differences disponível no Python.
* Analisar consistência, precisão, robustez e desempenho computacional dos dois 


!pip install differences
!pip install doubleml

In [28]:
import pandas as pd
import numpy as np

data = pd.read_stata("https://github.com/Daniel-Uhr/data/raw/main/bacon_example.dta")

In [30]:
# Filtragem dos dados
# Vamos criar identificadores estaduais
data['id'] = data['stfips'].rank(method='dense').astype(int)

# time to treat
data['t'] = data['year'] - data['_nfd']

# Outcome (Suicide Mortality)
data['y'] = data['asmrs']

# Covariáveis - pcinc asmrh cases
data['X1'] = data['pcinc']
data['X2'] = data['asmrh']
data['X3'] = data['cases']

# Treatment
data['D'] = data['post']
data['media_D'] = data.groupby('id')['D'].transform('mean')
data['always_treated'] = np.where(data['media_D'] == 1, 1, 0)
data['never_treated'] = np.where(data['media_D'] == 0, 1, 0)
# foi tratado em algum momento media_D for maior que 0, será um valor 1 para o id
data['treated'] = np.where(data['media_D'] > 0, 1, 0)

# Vamos criar a variável de grupo G
data['cohort']=data['_nfd']

In [31]:
print(sorted(data['t'].unique()))


[-21.0, -20.0, -19.0, -18.0, -17.0, -16.0, -15.0, -14.0, -13.0, -12.0, -11.0, -10.0, -9.0, -8.0, -7.0, -6.0, -5.0, -4.0, -3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, nan, 26.0, 27.0]


In [32]:
data

,stfips,year,_nfd,post,asmrs,pcinc,asmrh,cases,weight,copop,...,y,X1,X2,X3,D,media_D,always_treated,never_treated,treated,cohort
0,1,1964.0,1971.0,0.0,35.639885,12406.178537,5.007341,0.012312,1.715156e+06,1.715156e+06,...,35.639885,12406.178537,5.007341,0.012312,0.0,0.787879,0,0,1,1971.0
1,1,1965.0,1971.0,0.0,41.543755,13070.206738,4.425367,0.010419,1.715156e+06,1.725186e+06,...,41.543755,13070.206738,4.425367,0.010419,0.0,0.787879,0,0,1,1971.0
2,1,1966.0,1971.0,0.0,34.252335,13526.663217,4.874819,0.009900,1.715156e+06,1.735219e+06,...,34.252335,13526.663217,4.874819,0.009900,0.0,0.787879,0,0,1,1971.0
3,1,1967.0,1971.0,0.0,34.465023,13918.189823,5.362014,0.009975,1.715156e+06,1.745250e+06,...,34.465023,13918.189823,5.362014,0.009975,0.0,0.787879,0,0,1,1971.0
4,1,1968.0,1971.0,0.0,40.440105,14684.808682,4.643759,0.012401,1.715156e+06,1.755283e+06,...,40.440105,14684.808682,4.643759,0.012401,0.0,0.787879,0,0,1,1971.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1612,56,1992.0,1977.0,1.0,33.149574,31181.450546,1.970134,0.028996,1.621688e+05,2.323830e+05,...,33.149574,31181.450546,1.970134,0.028996,1.0,0.606061,0,0,1,1977.0
1613,56,1993.0,1977.0,1.0,63.909775,31666.776231,3.020953,0.026672,1.621688e+05,2.355600e+05,...,63.909775,31666.776231,3.020953,0.026672,1.0,0.606061,0,0,1,1977.0
1614,56,1994.0,1977.0,1.0,81.793816,31959.528356,3.501871,0.023643,1.621688e+05,2.389800e+05,...,81.793816,31959.528356,3.501871,0.023643,1.0,0.606061,0,0,1,1977.0
1615,56,1995.0,1977.0,1.0,30.630585,32327.659531,0.740606,0.020965,1.621688e+05,2.414090e+05,...,30.630585,32327.659531,0.740606,0.020965,1.0,0.606061,0,0,1,1977.0


In [33]:
data['cohort'].describe()

count    1188.000000
mean     1973.583333
std         3.539969
min      1969.000000
25%      1971.000000
50%      1973.000000
75%      1974.250000
max      1985.000000
Name: cohort, dtype: float64

In [34]:
# Criar uma base de dados com apenas um grupo tratado específico e todos os nunca tratados.
data_sz = data[(data['cohort'] == 1973) | (data['never_treated'] == 1)]

In [35]:
data_sz

,stfips,year,_nfd,post,asmrs,pcinc,asmrh,cases,weight,copop,...,y,X1,X2,X3,D,media_D,always_treated,never_treated,treated,cohort
33,4,1964.0,1973.0,0.0,73.528717,15551.315938,3.083218,0.012883,744813.0,7.448130e+05,...,73.528717,15551.315938,3.083218,0.012883,0.0,0.727273,0,0,1,1973.0
34,4,1965.0,1973.0,0.0,82.431198,15962.689260,3.005868,0.012225,744813.0,7.692093e+05,...,82.431198,15962.689260,3.005868,0.012225,0.0,0.727273,0,0,1,1973.0
35,4,1966.0,1973.0,0.0,59.255363,16581.270102,5.074290,0.012180,744813.0,7.936050e+05,...,59.255363,16581.270102,5.074290,0.012180,0.0,0.727273,0,0,1,1973.0
36,4,1967.0,1973.0,0.0,81.493416,17214.603202,3.836898,0.012327,744813.0,8.180007e+05,...,81.493416,17214.603202,3.836898,0.012327,0.0,0.727273,0,0,1,1973.0
37,4,1968.0,1973.0,0.0,72.936737,18610.393366,5.394385,0.012178,744813.0,8.423957e+05,...,72.936737,18610.393366,5.394385,0.012178,0.0,0.727273,0,0,1,1973.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1513,53,1992.0,1973.0,1.0,50.471672,35482.948403,3.107361,0.037566,1546064.0,2.598309e+06,...,50.471672,35482.948403,3.107361,0.037566,1.0,0.727273,0,0,1,1973.0
1514,53,1993.0,1973.0,1.0,47.067001,35571.542235,2.844220,0.038398,1546064.0,2.656901e+06,...,47.067001,35571.542235,2.844220,0.038398,1.0,0.727273,0,0,1,1973.0
1515,53,1994.0,1973.0,1.0,45.564236,35987.602571,3.035455,0.038123,1546064.0,2.705420e+06,...,45.564236,35987.602571,3.035455,0.038123,1.0,0.727273,0,0,1,1973.0
1516,53,1995.0,1973.0,1.0,45.268627,36422.286961,3.326494,0.036635,1546064.0,2.756447e+06,...,45.268627,36422.286961,3.326494,0.036635,1.0,0.727273,0,0,1,1973.0


#### Aplicar DoubleML-DID



Analisar a dupla diferença para os períodos t=-1 e t=1.

In [9]:
# 1. Manter na base apenas t == -1, t == 1 ou t == NA
data_sz_1 = data_sz[(data_sz['t'].isin([-1.0, 1.0])) | (data_sz['t'].isna())]

# 2. Identificar anos dos tratados
anos_tratados = data_sz_1[data_sz_1['never_treated'] == 0]['year'].unique()

# 3. Filtrar: manter tratados ou never_treated com anos dos tratados
data_sz_1 = data_sz_1[
    (data_sz_1['never_treated'] == 0) | 
    ((data_sz_1['never_treated'] == 1) & (data_sz_1['year'].isin(anos_tratados)))
]
data_sz_1

,stfips,year,_nfd,post,asmrs,pcinc,asmrh,cases,weight,copop,...,y,X1,X2,X3,D,media_D,always_treated,never_treated,treated,cohort
41,4,1972.0,1973.0,0.0,101.922516,22979.025315,3.912483,0.018428,7.448130e+05,1017302.0,...,101.922516,22979.025315,3.912483,0.018428,0.0,0.727273,0,0,1,1973.0
43,4,1974.0,1973.0,1.0,87.002365,23114.632475,4.767786,0.018148,7.448130e+05,1125678.0,...,87.002365,23114.632475,4.767786,0.018148,1.0,0.727273,0,0,1,1973.0
74,5,1972.0,NaN,0.0,56.250359,17225.678238,4.724695,0.020913,9.379597e+05,1038165.0,...,56.250359,17225.678238,4.724695,0.020913,0.0,0.000000,0,1,0,NaN
76,5,1974.0,NaN,0.0,48.406815,18717.644793,4.108365,0.026070,9.379597e+05,1080970.0,...,48.406815,18717.644793,4.108365,0.026070,0.0,0.000000,0,1,0,NaN
173,9,1972.0,1973.0,0.0,47.559387,28467.286435,1.722194,0.020316,1.407318e+06,1580484.0,...,47.559387,28467.286435,1.722194,0.020316,0.0,0.727273,0,0,1,1973.0
175,9,1974.0,1973.0,1.0,38.020470,28855.494139,2.411515,0.023150,1.407318e+06,1586262.0,...,38.020470,28855.494139,2.411515,0.023150,1.0,0.727273,0,0,1,1973.0
206,10,1972.0,NaN,0.0,47.521610,26803.135702,1.417574,0.030777,2.475022e+05,294246.0,...,47.521610,26803.135702,1.417574,0.030777,0.0,0.000000,0,1,0,NaN
208,10,1974.0,NaN,0.0,56.700481,27049.663476,5.338567,0.030429,2.475022e+05,299688.0,...,56.700481,27049.663476,5.338567,0.030429,0.0,0.000000,0,1,0,NaN
305,13,1972.0,1973.0,0.0,67.377106,20588.342698,7.584846,0.038746,2.150928e+06,2476205.0,...,67.377106,20588.342698,7.584846,0.038746,0.0,0.727273,0,0,1,1973.0
307,13,1974.0,1973.0,1.0,64.986893,21006.430168,9.746268,0.041766,2.150928e+06,2576772.0,...,64.986893,21006.430168,9.746268,0.041766,1.0,0.727273,0,0,1,1973.0


Fazer o procedimento para utilizar o DML

In [36]:
# criar uma variável de diferença de y na base data_sz_1['y_diff'] 
data_sz_1['y_diff'] = data_sz_1.groupby('id')['y'].diff()

# substituir valores data_sz_1['y_diff'] "na" pelo valor da y_diff do id calculado para o período posterior
data_sz_1['y_diff'] = data_sz_1.groupby('id')['y_diff'].fillna(method='bfill')


C:\Users\danie\AppData\Local\Temp\ipykernel_29616\1607195728.py:5: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  data_sz_1['y_diff'] = data_sz_1.groupby('id')['y_diff'].fillna(method='bfill')
C:\Users\danie\AppData\Local\Temp\ipykernel_29616\1607195728.py:5: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data_sz_1['y_diff'] = data_sz_1.groupby('id')['y_diff'].fillna(method='bfill')


In [37]:
data_sz_1

,stfips,year,_nfd,post,asmrs,pcinc,asmrh,cases,weight,copop,...,X3,D,media_D,always_treated,never_treated,treated,cohort,y_diff,year1,id1
41,4,1972.0,1973.0,0.0,101.922516,22979.025315,3.912483,0.018428,7.448130e+05,1017302.0,...,0.018428,0.0,0.727273,0,0,1,1973.0,-14.920151,1972,2
43,4,1974.0,1973.0,1.0,87.002365,23114.632475,4.767786,0.018148,7.448130e+05,1125678.0,...,0.018148,1.0,0.727273,0,0,1,1973.0,-14.920151,1974,2
74,5,1972.0,NaN,0.0,56.250359,17225.678238,4.724695,0.020913,9.379597e+05,1038165.0,...,0.020913,0.0,0.000000,0,1,0,NaN,-7.843544,1972,3
76,5,1974.0,NaN,0.0,48.406815,18717.644793,4.108365,0.026070,9.379597e+05,1080970.0,...,0.026070,0.0,0.000000,0,1,0,NaN,-7.843544,1974,3
173,9,1972.0,1973.0,0.0,47.559387,28467.286435,1.722194,0.020316,1.407318e+06,1580484.0,...,0.020316,0.0,0.727273,0,0,1,1973.0,-9.538918,1972,6
175,9,1974.0,1973.0,1.0,38.020470,28855.494139,2.411515,0.023150,1.407318e+06,1586262.0,...,0.023150,1.0,0.727273,0,0,1,1973.0,-9.538918,1974,6
206,10,1972.0,NaN,0.0,47.521610,26803.135702,1.417574,0.030777,2.475022e+05,294246.0,...,0.030777,0.0,0.000000,0,1,0,NaN,9.178871,1972,7
208,10,1974.0,NaN,0.0,56.700481,27049.663476,5.338567,0.030429,2.475022e+05,299688.0,...,0.030429,0.0,0.000000,0,1,0,NaN,9.178871,1974,7
305,13,1972.0,1973.0,0.0,67.377106,20588.342698,7.584846,0.038746,2.150928e+06,2476205.0,...,0.038746,0.0,0.727273,0,0,1,1973.0,-2.390213,1972,10
307,13,1974.0,1973.0,1.0,64.986893,21006.430168,9.746268,0.041766,2.150928e+06,2576772.0,...,0.041766,1.0,0.727273,0,0,1,1973.0,-2.390213,1974,10


In [38]:
df_dml = data_sz_1[data_sz_1['year'] == data_sz_1['year'].min()].copy()
df_dml

,stfips,year,_nfd,post,asmrs,pcinc,asmrh,cases,weight,copop,...,X3,D,media_D,always_treated,never_treated,treated,cohort,y_diff,year1,id1
41,4,1972.0,1973.0,0.0,101.922516,22979.025315,3.912483,0.018428,7.448130e+05,1017302.0,...,0.018428,0.0,0.727273,0,0,1,1973.0,-14.920151,1972,2
74,5,1972.0,NaN,0.0,56.250359,17225.678238,4.724695,0.020913,9.379597e+05,1038165.0,...,0.020913,0.0,0.000000,0,1,0,NaN,-7.843544,1972,3
173,9,1972.0,1973.0,0.0,47.559387,28467.286435,1.722194,0.020316,1.407318e+06,1580484.0,...,0.020316,0.0,0.727273,0,0,1,1973.0,-9.538918,1972,6
206,10,1972.0,NaN,0.0,47.521610,26803.135702,1.417574,0.030777,2.475022e+05,294246.0,...,0.030777,0.0,0.000000,0,1,0,NaN,9.178871,1972,7
305,13,1972.0,1973.0,0.0,67.377106,20588.342698,7.584846,0.038746,2.150928e+06,2476205.0,...,0.038746,0.0,0.727273,0,0,1,1973.0,-2.390213,1972,10
404,18,1972.0,1973.0,0.0,56.968685,22011.952470,2.517947,0.017422,2.490114e+06,2711777.0,...,0.017422,0.0,0.727273,0,0,1,1973.0,-5.073200,1972,13
569,23,1972.0,1973.0,0.0,46.754639,19694.904842,3.060293,0.034757,4.958082e+05,529513.0,...,0.034757,0.0,0.727273,0,0,1,1973.0,16.283157,1972,18
734,28,1972.0,NaN,0.0,43.245850,16283.150389,6.472214,0.037212,1.118302e+06,1186875.0,...,0.037212,0.0,0.000000,0,1,0,NaN,0.141403,1972,23
767,29,1972.0,1973.0,0.0,58.237740,22350.673416,3.024333,0.026119,2.298065e+06,2459347.0,...,0.026119,0.0,0.727273,0,0,1,1973.0,1.111126,1972,24
866,32,1972.0,1973.0,0.0,111.368187,28388.742447,5.085210,0.017197,1.795353e+05,269158.0,...,0.017197,0.0,0.727273,0,0,1,1973.0,49.472572,1972,27


In [39]:
X = df_dml[['X1', 'X2', 'X3']]
y = df_dml['y_diff']
d = df_dml['treated']


In [40]:
from doubleml import DoubleMLData

dml_data = DoubleMLData.from_arrays(
    x=X,
    y=y,
    d=d
)


In [41]:
from doubleml import DoubleMLDID
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

ml_g = RandomForestRegressor(max_depth=5)
ml_m = RandomForestClassifier(max_depth=5)

dml_did = DoubleMLDID(
    dml_data,
    ml_g=ml_g,
    ml_m=ml_m,
    score='observational',
    n_folds=2  # ou 1
)

dml_did.fit()


In [42]:
print(dml_did.summary)

print("Intervalo de confiança 95%:")
print(dml_did.confint(level=0.95))


       coef   std err         t     P>|t|      2.5 %     97.5 %
d -1.595184  7.558474 -0.211046  0.832851 -16.409521  13.219152
Intervalo de confiança 95%:
       2.5 %     97.5 %
d -16.409521  13.219152


In [43]:
df_dml

,stfips,year,_nfd,post,asmrs,pcinc,asmrh,cases,weight,copop,...,X3,D,media_D,always_treated,never_treated,treated,cohort,y_diff,year1,id1
41,4,1972.0,1973.0,0.0,101.922516,22979.025315,3.912483,0.018428,7.448130e+05,1017302.0,...,0.018428,0.0,0.727273,0,0,1,1973.0,-14.920151,1972,2
74,5,1972.0,NaN,0.0,56.250359,17225.678238,4.724695,0.020913,9.379597e+05,1038165.0,...,0.020913,0.0,0.000000,0,1,0,NaN,-7.843544,1972,3
173,9,1972.0,1973.0,0.0,47.559387,28467.286435,1.722194,0.020316,1.407318e+06,1580484.0,...,0.020316,0.0,0.727273,0,0,1,1973.0,-9.538918,1972,6
206,10,1972.0,NaN,0.0,47.521610,26803.135702,1.417574,0.030777,2.475022e+05,294246.0,...,0.030777,0.0,0.000000,0,1,0,NaN,9.178871,1972,7
305,13,1972.0,1973.0,0.0,67.377106,20588.342698,7.584846,0.038746,2.150928e+06,2476205.0,...,0.038746,0.0,0.727273,0,0,1,1973.0,-2.390213,1972,10
404,18,1972.0,1973.0,0.0,56.968685,22011.952470,2.517947,0.017422,2.490114e+06,2711777.0,...,0.017422,0.0,0.727273,0,0,1,1973.0,-5.073200,1972,13
569,23,1972.0,1973.0,0.0,46.754639,19694.904842,3.060293,0.034757,4.958082e+05,529513.0,...,0.034757,0.0,0.727273,0,0,1,1973.0,16.283157,1972,18
734,28,1972.0,NaN,0.0,43.245850,16283.150389,6.472214,0.037212,1.118302e+06,1186875.0,...,0.037212,0.0,0.000000,0,1,0,NaN,0.141403,1972,23
767,29,1972.0,1973.0,0.0,58.237740,22350.673416,3.024333,0.026119,2.298065e+06,2459347.0,...,0.026119,0.0,0.727273,0,0,1,1973.0,1.111126,1972,24
866,32,1972.0,1973.0,0.0,111.368187,28388.742447,5.085210,0.017197,1.795353e+05,269158.0,...,0.017197,0.0,0.727273,0,0,1,1973.0,49.472572,1972,27


In [44]:
data_sz_1

,stfips,year,_nfd,post,asmrs,pcinc,asmrh,cases,weight,copop,...,X3,D,media_D,always_treated,never_treated,treated,cohort,y_diff,year1,id1
41,4,1972.0,1973.0,0.0,101.922516,22979.025315,3.912483,0.018428,7.448130e+05,1017302.0,...,0.018428,0.0,0.727273,0,0,1,1973.0,-14.920151,1972,2
43,4,1974.0,1973.0,1.0,87.002365,23114.632475,4.767786,0.018148,7.448130e+05,1125678.0,...,0.018148,1.0,0.727273,0,0,1,1973.0,-14.920151,1974,2
74,5,1972.0,NaN,0.0,56.250359,17225.678238,4.724695,0.020913,9.379597e+05,1038165.0,...,0.020913,0.0,0.000000,0,1,0,NaN,-7.843544,1972,3
76,5,1974.0,NaN,0.0,48.406815,18717.644793,4.108365,0.026070,9.379597e+05,1080970.0,...,0.026070,0.0,0.000000,0,1,0,NaN,-7.843544,1974,3
173,9,1972.0,1973.0,0.0,47.559387,28467.286435,1.722194,0.020316,1.407318e+06,1580484.0,...,0.020316,0.0,0.727273,0,0,1,1973.0,-9.538918,1972,6
175,9,1974.0,1973.0,1.0,38.020470,28855.494139,2.411515,0.023150,1.407318e+06,1586262.0,...,0.023150,1.0,0.727273,0,0,1,1973.0,-9.538918,1974,6
206,10,1972.0,NaN,0.0,47.521610,26803.135702,1.417574,0.030777,2.475022e+05,294246.0,...,0.030777,0.0,0.000000,0,1,0,NaN,9.178871,1972,7
208,10,1974.0,NaN,0.0,56.700481,27049.663476,5.338567,0.030429,2.475022e+05,299688.0,...,0.030429,0.0,0.000000,0,1,0,NaN,9.178871,1974,7
305,13,1972.0,1973.0,0.0,67.377106,20588.342698,7.584846,0.038746,2.150928e+06,2476205.0,...,0.038746,0.0,0.727273,0,0,1,1973.0,-2.390213,1972,10
307,13,1974.0,1973.0,1.0,64.986893,21006.430168,9.746268,0.041766,2.150928e+06,2576772.0,...,0.041766,1.0,0.727273,0,0,1,1973.0,-2.390213,1974,10


In [45]:
# O pacote precisa entender a estrutura de painel dos dados. Precisamos ajustar a variável year para int (inteiro)
# Criando clones
data_sz_1['year1'] = data_sz_1['year']
data_sz_1['id1'] = data_sz_1['id']
data_sz_1['year1'] = data_sz_1['year'].astype(int)
data_sz_1

,stfips,year,_nfd,post,asmrs,pcinc,asmrh,cases,weight,copop,...,X3,D,media_D,always_treated,never_treated,treated,cohort,y_diff,year1,id1
41,4,1972.0,1973.0,0.0,101.922516,22979.025315,3.912483,0.018428,7.448130e+05,1017302.0,...,0.018428,0.0,0.727273,0,0,1,1973.0,-14.920151,1972,2
43,4,1974.0,1973.0,1.0,87.002365,23114.632475,4.767786,0.018148,7.448130e+05,1125678.0,...,0.018148,1.0,0.727273,0,0,1,1973.0,-14.920151,1974,2
74,5,1972.0,NaN,0.0,56.250359,17225.678238,4.724695,0.020913,9.379597e+05,1038165.0,...,0.020913,0.0,0.000000,0,1,0,NaN,-7.843544,1972,3
76,5,1974.0,NaN,0.0,48.406815,18717.644793,4.108365,0.026070,9.379597e+05,1080970.0,...,0.026070,0.0,0.000000,0,1,0,NaN,-7.843544,1974,3
173,9,1972.0,1973.0,0.0,47.559387,28467.286435,1.722194,0.020316,1.407318e+06,1580484.0,...,0.020316,0.0,0.727273,0,0,1,1973.0,-9.538918,1972,6
175,9,1974.0,1973.0,1.0,38.020470,28855.494139,2.411515,0.023150,1.407318e+06,1586262.0,...,0.023150,1.0,0.727273,0,0,1,1973.0,-9.538918,1974,6
206,10,1972.0,NaN,0.0,47.521610,26803.135702,1.417574,0.030777,2.475022e+05,294246.0,...,0.030777,0.0,0.000000,0,1,0,NaN,9.178871,1972,7
208,10,1974.0,NaN,0.0,56.700481,27049.663476,5.338567,0.030429,2.475022e+05,299688.0,...,0.030429,0.0,0.000000,0,1,0,NaN,9.178871,1974,7
305,13,1972.0,1973.0,0.0,67.377106,20588.342698,7.584846,0.038746,2.150928e+06,2476205.0,...,0.038746,0.0,0.727273,0,0,1,1973.0,-2.390213,1972,10
307,13,1974.0,1973.0,1.0,64.986893,21006.430168,9.746268,0.041766,2.150928e+06,2576772.0,...,0.041766,1.0,0.727273,0,0,1,1973.0,-2.390213,1974,10


In [ ]:
data_sz_1.set_index(['id1', 'year1'], inplace=True)

In [49]:
from differences import ATTgt

att_gt = ATTgt(data=data_sz_1, cohort_name='cohort')

att_gt.fit(formula='y ~ X1 + X2 + X3',)

att_gt.aggregate("cohort", overall=True)

Computing ATTgt [workers=1]     0%|                    | 0/1 [00:00<?, ?it/s]

Computing ATTgt [workers=1]   100%|████████████████████| 1/1 [00:00<00:00, 29.67it/s]


CohortAggregationOverall                                            \
                            analytic pointwise conf. band              
                       ATT std_error                lower      upper   
0                 3.098641  8.334399            -13.23648  19.433762   

                     
                     
  zero_not_in_cband  
0

In [ ]:
attgt.fit()
